In [1]:
# ============================================================
# BLOCO 01 — CONFIGURAÇÕES
# ============================================================

from pathlib import Path

import pandas as pd

ROOT = Path("../")

DADOS_PROCESSADOS = ROOT / "dados_processados"

ANOS = [str(ano) for ano in range(2015, 2025)]

In [2]:
# ============================================================
# BLOCO 02 — LEITURA DAS BASES
# ============================================================

bases = {
    "homicidios": pd.read_parquet(DADOS_PROCESSADOS / "homicidios.parquet"),
    "latrocinios": pd.read_parquet(DADOS_PROCESSADOS / "latrocinios.parquet"),
    "lesao_seguida_morte": pd.read_parquet(DADOS_PROCESSADOS / "lesao_seguida_morte.parquet"),
    "feminicidios": pd.read_parquet(DADOS_PROCESSADOS / "feminicidios.parquet"),
}

dim_regioes = pd.read_parquet(
    DADOS_PROCESSADOS / "dim_regioes.parquet"
)

print(f"Bases carregadas: {len(bases)}")
print(dim_regioes.shape)

Bases carregadas: 4
(36, 5)


In [3]:
# ============================================================
# BLOCO 03 — TRANSFORMAÇÃO PARA FORMATO LONGO
# ============================================================

bases_longas = {}

for nome, df in bases.items():

    bases_longas[nome] = df.melt(
        id_vars=[
            "regiao_administrativa",
            "nivel_geografico",
            "tipo_crime"
        ],
        value_vars=ANOS,
        var_name="ano",
        value_name="ocorrencias"
    )

print("Transformação concluída.")

Transformação concluída.


In [4]:
# ============================================================
# BLOCO 04 — CONSOLIDAÇÃO
# ============================================================

base = pd.concat(
    bases_longas.values(),
    ignore_index=True
)

base["ano"] = base["ano"].astype(int)

base["ocorrencias"] = (
    base["ocorrencias"]
    .fillna(0)
    .astype(int)
)

print(base.shape)

(1370, 5)


In [5]:
# ============================================================
# BLOCO 05 — INTEGRAÇÃO DA DIMENSÃO TERRITORIAL
# ============================================================

base = base.merge(
    dim_regioes,
    on="regiao_administrativa",
    how="left",
    validate="many_to_one"
)

print(base.shape)

(1370, 9)


In [6]:
# ============================================================
# BLOCO 05.1 — TAXAS PADRONIZADAS
# ============================================================

base["taxa_100mil"] = (
    base["ocorrencias"] / base["populacao"] * 100_000
)

base["taxa_10mil"] = (
    base["ocorrencias"] / base["populacao"] * 10_000
)

# Arredondamento para facilitar análises e visualizações
base["taxa_100mil"] = base["taxa_100mil"].round(2)
base["taxa_10mil"] = base["taxa_10mil"].round(2)

In [7]:
# ==============================================================================
# BLOCO 05.2 — VARIÁVEIS TEMPORAIS
# ==============================================================================

# Ordenação cronológica
base = (
    base
    .sort_values(
        ["nivel_geografico",
         "regiao_administrativa",
         "tipo_crime",
         "ano"]
    )
    .reset_index(drop=True)
)

# Ano inicial da série
ANO_INICIAL = base["ano"].min()

# Anos transcorridos desde o início da série
base["anos_desde_inicio"] = (
    base["ano"] - ANO_INICIAL
)

# Índice temporal (2015 = 100)
base["indice_tempo"] = (
    100 + base["anos_desde_inicio"] * 100
)

In [8]:
# ==============================================================================
# BLOCO 05.3 — VARIAÇÕES ANUAIS
# ==============================================================================

agrupamento = [
    "nivel_geografico",
    "regiao_administrativa",
    "tipo_crime"
]

base["delta_ocorrencias"] = (
    base
    .groupby(agrupamento)["ocorrencias"]
    .diff()
)

base["delta_taxa_100mil"] = (
    base
    .groupby(agrupamento)["taxa_100mil"]
    .diff()
)

base["delta_taxa_10mil"] = (
    base
    .groupby(agrupamento)["taxa_10mil"]
    .diff()
)

In [9]:
# ==============================================================================
# BLOCO 05.3 — CRESCIMENTO PERCENTUAL
# ==============================================================================

base["variacao_pct_ocorrencias"] = (
    base
    .groupby(agrupamento)["ocorrencias"]
    .pct_change() * 100
)

base["variacao_pct_taxa_100mil"] = (
    base
    .groupby(agrupamento)["taxa_100mil"]
    .pct_change() * 100
)

base["variacao_pct_taxa_10mil"] = (
    base
    .groupby(agrupamento)["taxa_10mil"]
    .pct_change() * 100
)

In [10]:
# ============================================================
# BLOCO 06 — VALIDAÇÃO
# ============================================================

print("=" * 70)

print("Registros:", len(base))

print("\nValores ausentes")

print(base.isna().sum())

print("\nTipos")

print(base.dtypes)

print("\nAnos")

print(sorted(base["ano"].unique()))

print("\nTipos de crime")

print(base["tipo_crime"].value_counts())

Registros: 1370

Valores ausentes
regiao_administrativa         0
nivel_geografico              0
tipo_crime                    0
ano                           0
ocorrencias                   0
populacao                    10
renda_per_capita             10
renda_domiciliar             50
fonte                        10
taxa_100mil                  10
taxa_10mil                   10
anos_desde_inicio             0
indice_tempo                  0
delta_ocorrencias           137
delta_taxa_100mil           146
delta_taxa_10mil            146
variacao_pct_ocorrencias    644
variacao_pct_taxa_100mil    646
variacao_pct_taxa_10mil     646
dtype: int64

Tipos
regiao_administrativa        object
nivel_geografico             object
tipo_crime                   object
ano                           int64
ocorrencias                   int64
populacao                   float64
renda_per_capita            float64
renda_domiciliar            float64
fonte                        object
taxa_100mil   

In [11]:
# ============================================================
# BLOCO 06.1 — DIAGNÓSTICO DA INTEGRAÇÃO COM A DIMENSÃO
# ============================================================

print("=" * 80)
print("REGIÕES SEM CORRESPONDÊNCIA")
print("=" * 80)

# Regiões da base analítica sem população/renda
sem_dim = (
    base.loc[
        base["populacao"].isna(),
        "regiao_administrativa"
    ]
    .drop_duplicates()
    .sort_values()
)

print(f"\nRegiões da base sem correspondência: {len(sem_dim)}")

if len(sem_dim):
    print(sem_dim.to_list())
else:
    print("Nenhuma.")


print("\n" + "=" * 80)
print("REGIÕES EXISTENTES NA DIMENSÃO, MAS AUSENTES NA BASE")
print("=" * 80)

sem_base = (
    dim_regioes.loc[
        ~dim_regioes["regiao_administrativa"].isin(
            base["regiao_administrativa"]
        ),
        "regiao_administrativa"
    ]
    .sort_values()
)

print(f"\nRegiões não utilizadas: {len(sem_base)}")

if len(sem_base):
    print(sem_base.to_list())
else:
    print("Nenhuma.")


print("\n" + "=" * 80)
print("RESUMO DA COBERTURA")
print("=" * 80)

print(f"Regiões na base analítica : {base['regiao_administrativa'].nunique()}")
print(f"Regiões na dimensão       : {dim_regioes['regiao_administrativa'].nunique()}")

print(f"\nRegistros totais          : {len(base)}")
print(f"Registros sem população   : {base['populacao'].isna().sum()}")
print(f"Registros sem renda       : {base['renda_per_capita'].isna().sum()}")

print(
    f"\nCobertura da dimensão: "
    f"{100 * (1 - base['populacao'].isna().mean()):.2f}%"
)

REGIÕES SEM CORRESPONDÊNCIA

Regiões da base sem correspondência: 1
['Unidades Prisionais']

REGIÕES EXISTENTES NA DIMENSÃO, MAS AUSENTES NA BASE

Regiões não utilizadas: 2
['Arapoanga', 'Água Quente']

RESUMO DA COBERTURA
Regiões na base analítica : 35
Regiões na dimensão       : 36

Registros totais          : 1370
Registros sem população   : 10
Registros sem renda       : 10

Cobertura da dimensão: 99.27%


In [12]:
# ============================================================
# BLOCO 07 — EXPORTAÇÃO
# ============================================================

base.to_parquet(
    DADOS_PROCESSADOS / "base_analitica.parquet",
    index=False
)

print("Base analítica exportada com sucesso.")

Base analítica exportada com sucesso.
